# Notebook 1 — Gamry Import & Validation (v1.0)

Imports and validates the complete WE-vs-Hg/HgO polarization experiment, reconstructs 20 CP–EIS–CV steps, selects the last complete CV cycle, averages initial/final EIS replicates, and exports data for Notebook 2.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

## 1. Configuration — edit this cell only

In [ ]:
EXPERIMENT_NAME = "Stability_Pol-curve_Ni-mesh_7M_KOH"
RAW_DATA_DIR = Path(r"C:\Users\edr2299\OneDrive - The University of Texas at Austin\Documents\Research\Electrochemistry Analysis\Data\Stability_Pol-curve_Ni-mesh_7M_KOH")
OUTPUT_DIR = Path(r"C:\Users\edr2299\OneDrive - The University of Texas at Austin\Documents\Research\Electrochemistry Analysis\Results\Processed_data_Notebook_1")
META = {"experiment_name":EXPERIMENT_NAME,"working_electrode":"Ni mesh","counter_electrode":"Ni mesh","reference_electrode":"Hg/HgO","reference_filling_solution":"7 M KOH","electrolyte":"7 M KOH","koh_condition":"Fe-unpurified","temperature_C":25.0,"flow_rate_mL_min":100.0,"geometric_area_cm2":4.0}
EXPECTED = {"steps":20,"activation_A":0.060,"activation_s":28800,"preconditioning_A":0.004,"cp_s":600,"eis_initial_Hz":100000,"eis_step_final_Hz":0.8,"eis_replicate_final_Hz":0.05,"eis_ppd":10}
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(OUTPUT_DIR/'tables').mkdir(exist_ok=True)
(OUTPUT_DIR/'selected_cv_cycles').mkdir(exist_ok=True)
print('Raw:',RAW_DATA_DIR,'\nExists:',RAW_DATA_DIR.exists(),'\nOutput:',OUTPUT_DIR)

## 2. Parser, protocol reconstruction, and validation

In [ ]:
def text(path):
    for enc in ('utf-8-sig','utf-8','cp1252','latin-1'):
        try:return path.read_text(encoding=enc)
        except UnicodeDecodeError:pass
    raise UnicodeError(path)

def val(x):
    if isinstance(x,dict):x=x.get('value')
    try:return float(x)
    except:return x

def hget(h,k,d=None):return val(h.get(k,d))
def marker(line):return bool(re.fullmatch(r'(CURVE\d*|ZCURVE\d*|OCVCURVE\d*|TABLE\d*)',line.split('\t',1)[0].strip(),re.I))

def parse(path):
    lines=text(path).splitlines(); h={}; tables={}; i=0
    while i<len(lines):
        if marker(lines[i]):
            name=lines[i].split('\t',1)[0].strip(); ci=None
            for j in range(i+1,min(i+8,len(lines))):
                cols=[x.strip() for x in lines[j].lstrip('\t').split('\t')]
                if len(set(cols)&{'Pt','T','Time','Vf','Im','Freq','Zreal','Zimag','Idc','Vdc'})>=2:ci=j;break
            if ci is None:i+=1;continue
            cols=[x.strip() for x in lines[ci].lstrip('\t').split('\t')]; rows=[]; j=ci+1
            while j<len(lines):
                if marker(lines[j]):break
                p=lines[j].lstrip('\t').split('\t')
                try:float(p[0]);break
                except:j+=1
            while j<len(lines) and not marker(lines[j]):
                p=lines[j].lstrip('\t').split('\t')
                try:float(p[0])
                except:break
                rows.append((p+['']*len(cols))[:len(cols)]);j+=1
            df=pd.DataFrame(rows,columns=cols)
            for c in df:
                n=pd.to_numeric(df[c],errors='coerce')
                if len(df) and n.notna().mean()>=.75:df[c]=n
            if len(df):tables[name]=df
            i=j;continue
        p=lines[i].split('\t')
        if len(p)>=3 and p[0].strip():
            raw=p[2].strip()
            try:raw=float(raw)
            except:raw=True if raw in ('T','TRUE') else False if raw in ('F','FALSE') else raw
            h[p[0].strip()]={'value':raw,'description':p[3].strip() if len(p)>3 else ''}
        i+=1
    return h,tables

def classify(path):
    s=path.stem.lower().replace('-','_')
    fixed={'ocp_pre':('ocp_pre','ocp',None),'cpact':('activation_cp','cp',None),'cp_1ma_pre':('preconditioning_cp','cp',None),'cv_initial':('initial_cv','cv',None),'cv_final':('final_cv','cv',None)}
    if s in fixed:return fixed[s]
    m=re.fullmatch(r'eis_initial_#(\d+)',s)
    if m:return 'initial_eis','eis',int(m.group(1))
    m=re.fullmatch(r'eis_final_#(\d+)',s)
    if m:return 'final_eis','eis',int(m.group(1))
    patterns=[('pc_lc_1ma','polarization_cp','cp',1),('eis_pc_1ma','polarization_eis','eis',1),('cv_cycle_pc_1ma','polarization_cv','cv',1)]
    for p,r,f,n in patterns:
        if s==p:return r,f,n
    for prefix,r,f,base in [('pc_lc_ma_#','polarization_cp','cp',1),('eis_pc_ma_#','polarization_eis','eis',1),('cv_cycle_pc_ma_#','polarization_cv','cv',1),('pc_lc_#','polarization_cp','cp',5),('eis_pc_#','polarization_eis','eis',5),('cv_cycle_pc_#','polarization_cv','cv',5)]:
        m=re.fullmatch(re.escape(prefix[:-1])+r'(\d+)',s)
        if m:return r,f,base+int(m.group(1))
    return 'unclassified','unknown',None

def current(h,t,family):
    for k in (['IDCREQ'] if family=='eis' else ['ISTEP1']):
        x=hget(h,k)
        if isinstance(x,float):return x
    for df in t.values():
        for c in ('Idc','Im'):
            if c in df:
                x=pd.to_numeric(df[c],errors='coerce').dropna()
                if len(x):return float(x.iloc[int(.1*len(x)):].median())

def cv_complete(df,h):
    if 'Vf' not in df or len(df)<20:return False
    v=pd.to_numeric(df.Vf,errors='coerce').dropna().to_numpy(); tol=.015
    hi=hget(h,'VLIMIT1'); lo=hget(h,'VLIMIT2'); ini=hget(h,'VINIT')
    return len(v)>20 and (hi is None or v.max()>=hi-tol) and (lo is None or v.min()<=lo+tol) and np.any(np.diff(v)>0) and np.any(np.diff(v)<0) and (ini is None or abs(v[-1]-ini)<=2*tol)

def select_cv(t,h):
    good=[n for n,d in t.items() if cv_complete(d,h)]
    if good:return good[-1]
    names=list(t); good=[]
    for i in range(len(names)-1):
        if cv_complete(pd.concat([t[names[i]],t[names[i+1]]],ignore_index=True),h):good.append(names[i]+'+'+names[i+1])
    return good[-1] if good else None

def cv_df(item):
    s=item['selected_cv']
    return pd.concat([item['tables'][n] for n in s.split('+')],ignore_index=True) if '+' in s else item['tables'][s].copy()

if not RAW_DATA_DIR.exists():raise FileNotFoundError(f'Edit RAW_DATA_DIR: {RAW_DATA_DIR}')
files=sorted(RAW_DATA_DIR.rglob('*.DTA'),key=lambda p:p.name.lower())
if not files:raise AssertionError(f'No .DTA files found in {RAW_DATA_DIR}')
items=[]; parse_errors=[]
for p in files:
    try:
        h,t=parse(p); role,fam,seq=classify(p); area=hget(h,'AREA',META['geometric_area_cm2']); I=current(h,t,fam)
        items.append({'filename':p.name,'path':str(p),'role':role,'family':fam,'sequence':seq,'header':h,'tables':t,'current_A':I,'j_mA_cm2':None if I is None else I*1000/area,'selected_cv':select_cv(t,h) if fam=='cv' else None,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
    except Exception as e:parse_errors.append({'filename':p.name,'error':repr(e)})
print(f'Found {len(files)} raw Gamry files; parsed {len(items)}; errors {len(parse_errors)}')

def role(r):return [x for x in items if x['role']==r]
def indexed(r):return {x['sequence']:x for x in role(r)}
cp,eis,cv=indexed('polarization_cp'),indexed('polarization_eis'),indexed('polarization_cv')
steps=pd.DataFrame([{'step':n,'current_A':cp.get(n,eis.get(n,{})).get('current_A'),'current_density_mA_cm2':cp.get(n,eis.get(n,{})).get('j_mA_cm2'),'cp_file':cp.get(n,{}).get('filename'),'eis_file':eis.get(n,{}).get('filename'),'cv_file':cv.get(n,{}).get('filename'),'selected_cv_table':cv.get(n,{}).get('selected_cv')} for n in range(1,21)])
display(steps)

findings=[]
def rec(level,check,file,msg):findings.append({'level':level,'check':check,'filename':file,'message':msg})
for e in parse_errors:rec('ERROR','parse',e['filename'],e['error'])
for r in ('ocp_pre','activation_cp','initial_cv','preconditioning_cp','final_cv'):
    if len(role(r))!=1:rec('ERROR','protocol completeness','',f'{r}: expected 1, found {len(role(r))}')
for r in ('initial_eis','final_eis'):
    if len(role(r))!=3:rec('ERROR','replicate count','',f'{r}: expected 3, found {len(role(r))}')
for n,row in steps.iterrows():
    for c,label in [('cp_file','CP'),('eis_file','EIS'),('cv_file','CV')]:
        if pd.isna(row[c]):rec('ERROR','missing step file','',f"Step {row['step']}: {label}")
for x in items:
    a=hget(x['header'],'AREA')
    if a is not None and not np.isclose(a,META['geometric_area_cm2']):rec('WARNING','area',x['filename'],f'{a} cm²')
for n in range(1,21):
    if n in cp and n in eis and not np.isclose(cp[n]['current_A'],eis[n]['current_A'],rtol=.03,atol=5e-6):rec('ERROR','CP/EIS current',eis[n]['filename'],f"{cp[n]['current_A']} vs {eis[n]['current_A']} A")
    if n in cp and not np.isclose(hget(cp[n]['header'],'TSTEP1'),600,rtol=.03):rec('WARNING','CP duration',cp[n]['filename'],str(hget(cp[n]['header'],'TSTEP1')))
for x in role('initial_eis')+role('polarization_eis')+role('final_eis'):
    ef=EXPECTED['eis_step_final_Hz'] if x['role']=='polarization_eis' else EXPECTED['eis_replicate_final_Hz']
    for k,e in [('FREQINIT',100000),('FREQFINAL',ef),('PTSPERDEC',10)]:
        if not np.isclose(hget(x['header'],k),e,rtol=.05):rec('WARNING','EIS '+k,x['filename'],f"expected {e}, found {hget(x['header'],k)}")
for x in role('initial_cv')+role('polarization_cv')+role('final_cv'):
    cycles=2 if x['role']=='polarization_cv' else 3
    rate=5 if x['role']!='polarization_cv' or x['sequence'] in {2,3,4,5} else 50
    if round(hget(x['header'],'CYCLES'))!=cycles:rec('WARNING','CV cycles',x['filename'],f'expected {cycles}')
    if not np.isclose(hget(x['header'],'SCANRATE'),rate,rtol=.01):rec('WARNING','CV scan rate',x['filename'],f'expected {rate}')
    if not x['selected_cv']:rec('ERROR','last complete CV',x['filename'],'not detected')
if role('ocp_pre'):
    x=role('ocp_pre')[0]; df=next(iter(x['tables'].values())); end=pd.to_numeric(df.T,errors='coerce').max(); timeout=hget(x['header'],'TIMEOUT')
    if end<.9*timeout:rec('INFO','OCP completion',x['filename'],f'ended at {end:.1f} s after meeting stability criterion')
for x in role('unclassified'):rec('WARNING','unclassified',x['filename'],'filename not recognized')
validation=pd.DataFrame(findings or [{'level':'INFO','check':'validation','filename':'','message':'No findings'}])
display(validation)

## 3. EIS replicate summaries and export

In [ ]:
def eis_table(x):
    return next(d for d in x['tables'].values() if {'Freq','Zreal','Zimag'}.issubset(d.columns)).sort_values('Freq',ascending=False).reset_index(drop=True)
def eis_summary(group):
    long=[]
    for r,x in enumerate(sorted(group,key=lambda z:z['sequence']),1):
        d=eis_table(x)[['Freq','Zreal','Zimag']].copy();d['replicate']=r;d['rank']=range(1,len(d)+1);long.append(d)
    d=pd.concat(long);return d.groupby('rank',as_index=False).agg(frequency_Hz_mean=('Freq','mean'),frequency_Hz_sd=('Freq','std'),Zreal_ohm_mean=('Zreal','mean'),Zreal_ohm_sd=('Zreal','std'),Zimag_ohm_mean=('Zimag','mean'),Zimag_ohm_sd=('Zimag','std'),n=('replicate','nunique'))
initial_eis=eis_summary(role('initial_eis'));final_eis=eis_summary(role('final_eis'))
for x in items:
    exported=[]
    for name,d in x['tables'].items():
        q=OUTPUT_DIR/'tables'/f"{Path(x['filename']).stem}__{name}.csv";d.to_csv(q,index=False);exported.append(str(q.relative_to(OUTPUT_DIR)))
    x['table_csv_files']=exported
    if x['family']=='cv' and x['selected_cv']:
        q=OUTPUT_DIR/'selected_cv_cycles'/f"{Path(x['filename']).stem}__last_complete_cycle.csv";cv_df(x).to_csv(q,index=False);x['selected_cv_csv']=str(q.relative_to(OUTPUT_DIR))
steps.to_csv(OUTPUT_DIR/'polarization_step_overview.csv',index=False);validation.to_csv(OUTPUT_DIR/'validation_report.csv',index=False)
initial_eis.to_csv(OUTPUT_DIR/'initial_eis_mean_sd.csv',index=False);final_eis.to_csv(OUTPUT_DIR/'final_eis_mean_sd.csv',index=False)
pd.DataFrame([{'field':k,'value':v} for k,v in META.items()]).to_csv(OUTPUT_DIR/'experiment_metadata.csv',index=False)
manifest={'schema_version':'1.0','created_utc':datetime.now(timezone.utc).isoformat(timespec='seconds'),'metadata':META,'expected_protocol':EXPECTED,'raw_data_directory':str(RAW_DATA_DIR),'output_directory':str(OUTPUT_DIR),'raw_file_count':len(files),'files':[{k:v for k,v in x.items() if k!='tables'} for x in items],'polarization_steps':steps.where(pd.notna(steps),None).to_dict('records'),'validation_counts':validation.level.value_counts().to_dict()}
with open(OUTPUT_DIR/'experiment_manifest.json','w',encoding='utf-8') as f:json.dump(manifest,f,indent=2,ensure_ascii=False)
counts=validation.level.value_counts().to_dict();status='READY FOR NOTEBOOK 2' if counts.get('ERROR',0)==0 else 'REVIEW REQUIRED'
summary=pd.DataFrame({'Item':['Status','Raw .DTA files','Parsed files','Complete CP–EIS–CV steps','Initial EIS replicates','Final EIS replicates','CVs with selected last cycle','Errors','Warnings','Information','Output directory'],'Value':[status,len(files),len(items),len(steps.dropna(subset=['cp_file','eis_file','cv_file'])),len(role('initial_eis')),len(role('final_eis')),sum(x['family']=='cv' and x['selected_cv'] for x in items),counts.get('ERROR',0),counts.get('WARNING',0),counts.get('INFO',0),str(OUTPUT_DIR)]})
display(summary);display(Markdown('## ✅ Experiment reconstructed and exported' if status.startswith('READY') else '## ⚠️ Review validation errors before Notebook 2'))